# 07 — Kafka: события и стриминговый ингест в лейкхаус

Нужны профили `kafka` и `clickhouse` (для последнего раздела):
`make up-kafka`, `make up-clickhouse` — или `make up-full`.

Что здесь происходит:
1. шлём JSON-события продаж в топик `sales_events` (producer);
2. подглядываем в топик глазами консьюмера;
3. Spark Structured Streaming дочитывает бэклог топика в Iceberg-таблицу
   и завершается (`trigger(availableNow)`) — «стриминг по расписанию»;
4. ClickHouse подписывается на тот же топик сам — `ENGINE = Kafka` +
   materialized view.

Адреса брокера: изнутри docker-сети — `kafka:9092`, с хоста — `localhost:29092`.
Топики и consumer-группы удобно смотреть в kafka-ui: http://localhost:8082

In [ ]:
!pip install -q confluent-kafka clickhouse-connect

## 1. Producer: шлём события

Событие — той же формы, что строки таблицы `sales` в Greenplum. Ключ —
`region`: события одного региона попадают в одну партицию (= сохраняют
порядок между собой). То же самое делает скрипт
`scripts/kafka_produce_sales.py` и таска `produce_events` DAG'а
`kafka_pipeline`.

In [ ]:
import json
import random
from datetime import datetime, timedelta

from confluent_kafka import Producer

REGIONS = ["north", "south", "east", "west"]
PRODUCTS = ["widget", "gadget", "gizmo", "doohickey"]

producer = Producer({"bootstrap.servers": "kafka:9092"})
base_ts = datetime.now().replace(microsecond=0)

N = 100
for i in range(1, N + 1):
    event = {
        "id": i,
        "ts": (base_ts + timedelta(seconds=i)).isoformat(timespec="seconds"),
        "region": random.choice(REGIONS),
        "product": random.choice(PRODUCTS),
        "amount": round(random.uniform(1, 1000), 2),
    }
    producer.produce("sales_events", key=event["region"], value=json.dumps(event))
    producer.poll(0)          # отдаём delivery-callbacks, не копим очередь

producer.flush(30)
print(f"отправлено {N} событий в sales_events")

## 2. Consumer: подглядываем в топик

Сообщения из Kafka не исчезают при чтении — каждый консьюмер двигает свой
offset, и сколько консьюмеров подпишется, столько раз данные и прочитаются.
Здесь читаем первые три сообщения отдельной группой `notebook-peek`.

In [ ]:
from confluent_kafka import Consumer, TopicPartition

consumer = Consumer({
    "bootstrap.servers": "kafka:9092",
    "group.id": "notebook-peek",
    "auto.offset.reset": "earliest",   # новая группа начинает с начала топика
    "enable.auto.commit": False,       # offset не коммитим — это просто подглядывание
})

# сколько всего сообщений в топике: разница watermark-offset'ов по партициям
meta = consumer.list_topics("sales_events", timeout=15)
parts = meta.topics["sales_events"].partitions
total = 0
for p in parts:
    lo, hi = consumer.get_watermark_offsets(TopicPartition("sales_events", p), timeout=15)
    total += hi - lo
print(f"партиций: {len(parts)}, всего сообщений: {total}")

consumer.subscribe(["sales_events"])
got = 0
while got < 3:
    msg = consumer.poll(10)
    if msg is None or msg.error():
        continue
    print(msg.partition(), msg.offset(), json.loads(msg.value()))
    got += 1
consumer.close()

## 3. Spark Structured Streaming → Iceberg

`trigger(availableNow=True)` — джоба дочитывает всё, что накопилось в топике,
и завершается. Прогресс хранится в checkpoint'е на S3, поэтому повторный
запуск возьмёт только новые события — exactly-once в Iceberg без дублей.
Это та же логика, что в `jobs/kafka_to_iceberg.py` и DAG'е `kafka_pipeline`.

Коннектор Kafka не входит в образ — тянем через `spark.jars.packages`
(версия = версии Spark, при первом запуске нужен интернет).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import (DoubleType, IntegerType, StringType,
                               StructField, StructType, TimestampType)

spark = (
    SparkSession.builder.appName("07_kafka")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5")
    .getOrCreate()
)

event_schema = StructType([
    StructField("id", IntegerType()),
    StructField("ts", TimestampType()),
    StructField("region", StringType()),
    StructField("product", StringType()),
    StructField("amount", DoubleType()),
])

spark.sql("CREATE DATABASE IF NOT EXISTS hive.analytics")
spark.sql("""
    CREATE TABLE IF NOT EXISTS hive.analytics.sales_events (
        id INT, ts TIMESTAMP, region STRING, product STRING, amount DOUBLE,
        kafka_partition INT, kafka_offset BIGINT
    ) USING iceberg
""")

raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "sales_events")
    .option("startingOffsets", "earliest")
    .load()
)

events = (
    raw.select(
        from_json(col("value").cast("string"), event_schema).alias("e"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
    )
    .select("e.*", "kafka_partition", "kafka_offset")
)

q = (
    events.writeStream.format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "s3a://warehouse/_checkpoints/sales_events")
    .trigger(availableNow=True)
    .toTable("hive.analytics.sales_events")
)
q.awaitTermination()
print("строк в таблице:", spark.table("hive.analytics.sales_events").count())

Запусти ячейку с producer'ом ещё раз и повтори стриминг — в таблицу
доедут ровно новые события. История дозаписей видна в снапшотах Iceberg:

In [ ]:
spark.sql("""
    SELECT committed_at, operation,
           summary['added-records']  AS added,
           summary['total-records']  AS total
    FROM hive.analytics.sales_events.snapshots
    ORDER BY committed_at
""").show(truncate=False)

## 4. ClickHouse как стриминг-консьюмер: ENGINE = Kafka + MV

Spark здесь вообще не нужен: ClickHouse сам подписывается на топик.
Связка из трёх объектов:
- таблица `ENGINE = Kafka` — «окно» в топик (из неё нельзя селектить дважды);
- обычная MergeTree-таблица — куда складываем;
- materialized view — насос: перекладывает всё, что появляется в очереди.

Своя consumer-группа `clickhouse` стартует с начала топика и дальше
догоняет новые сообщения в реальном времени.

In [ ]:
import clickhouse_connect

ch = clickhouse_connect.get_client(host="clickhouse", port=8123,
                                   username="default", password="clickhouse")

ch.command("CREATE DATABASE IF NOT EXISTS kafka_demo")
ch.command("""
    CREATE TABLE IF NOT EXISTS kafka_demo.sales_events_queue (
        id Int32, ts DateTime, region String, product String, amount Float64
    ) ENGINE = Kafka
    SETTINGS kafka_broker_list = 'kafka:9092',
             kafka_topic_list = 'sales_events',
             kafka_group_name = 'clickhouse',
             kafka_format = 'JSONEachRow',
             date_time_input_format = 'best_effort'
""")
ch.command("""
    CREATE TABLE IF NOT EXISTS kafka_demo.sales_events (
        id Int32, ts DateTime, region String, product String, amount Float64
    ) ENGINE = MergeTree ORDER BY (region, ts)
""")
ch.command("""
    CREATE MATERIALIZED VIEW IF NOT EXISTS kafka_demo.sales_events_mv
    TO kafka_demo.sales_events AS
    SELECT * FROM kafka_demo.sales_events_queue
""")
print("OK: queue + mergetree + mv")

In [ ]:
import time

# шлём ещё пачку и смотрим, как MergeTree пополняется без нашего участия
before = ch.query("SELECT count() FROM kafka_demo.sales_events").result_rows[0][0]
for i in range(50):
    producer.produce("sales_events", value=json.dumps({
        "id": 9000 + i, "ts": datetime.now().isoformat(timespec="seconds"),
        "region": random.choice(REGIONS), "product": random.choice(PRODUCTS),
        "amount": round(random.uniform(1, 1000), 2),
    }))
producer.flush(30)

# MV забирает сообщения пачками раз в несколько секунд. Ждём не фиксированный
# sleep (на медленной машине может не хватить), а пока все 50 реально доедут.
deadline = time.time() + 60
while time.time() < deadline:
    if ch.query("SELECT count() FROM kafka_demo.sales_events").result_rows[0][0] >= before + 50:
        break
    time.sleep(2)
ch.query_df("SELECT count() rows, max(ts) latest FROM kafka_demo.sales_events")

## Итог

Один топик — три независимых читателя, каждый со своим offset'ом:

| Консьюмер | Группа | Куда пишет | Когда работает |
|---|---|---|---|
| Spark Structured Streaming | `spark-kafka-source-*` | Iceberg `analytics.sales_events` | по запуску (availableNow) |
| ClickHouse MV | `clickhouse` | MergeTree `kafka_demo.sales_events` | постоянно, сам |
| Ноутбук (peek) | `notebook-peek` | никуда | по запуску |

Оркестрованная версия первого пути — DAG `kafka_pipeline` в Airflow:
produce → spark-submit → сверка «сообщений в топике == строк в Iceberg».

Дальше: NiFi (см. README) умеет четвёртый путь — забирать тот же топик
визуальным flow и складывать сырые JSON в MinIO.